# Modelagem — House Prices

**Objetivo:** Treinar e comparar modelos de Aprendizado de Máquina usando o dataset pré-processado (`treino_limpo.csv`).  
**Métrica:** RMSLE — como a variável alvo já está em escala `log1p`, o RMSE calculado aqui é matematicamente equivalente ao RMSLE na escala original de dólares.  
**Modelos avaliados:** Ridge (baseline linear), Random Forest (ensemble), Gradient Boosting (boosting).

## 0. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42

## 1. Carregamento dos Dados

> O `treino_limpo.csv` foi gerado pelo `01_eda.ipynb`. Já contém: outliers removidos, nulos tratados, feature engineering aplicado, encoding ordinal e One-Hot, e `log1p` na variável alvo.

In [ ]:
df = pd.read_csv('../data/treino_limpo.csv')
print(f'Shape: {df.shape}')
print(f'Colunas numericas: {df.select_dtypes(include=np.number).shape[1]}')
print(f'Primeiras colunas: {df.columns[:8].tolist()}')

## 2. Separação X / y

A variável alvo `SalePrice` já está em escala `log1p(SalePrice_original)`.  
O modelo é treinado nessa escala. Para obter preços em dólares após a predição, basta aplicar `np.expm1()`.

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']  # log1p(SalePrice_original)

print(f'Features (X): {X.shape}')
print(f'Alvo (y):     {y.shape}')
print(f'Preco mediano estimado: ${np.expm1(y.median()):,.0f}')
print(f'Preco maximo estimado:  ${np.expm1(y.max()):,.0f}')
print(f'Assimetria do alvo:     {y.skew():.4f} (proximo de 0 = bom)')

## 3. Função de Avaliação — RMSLE via Validação Cruzada

**Por que RMSE = RMSLE aqui?**  
RMSLE(y_real, y_pred) = sqrt(mean( (log(y_real+1) - log(y_pred+1))² ))  
Como y já é log1p(y_real), isso é exatamente RMSE(y, y_pred_log).

Usamos `KFold` com 5 dobras, embaralhamento e seed fixo para garantir reprodutibilidade.

In [ ]:
def rmsle_cv(modelo, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    scores = -cross_val_score(
        modelo, X, y,
        scoring='neg_root_mean_squared_error',
        cv=kf, n_jobs=-1
    )
    return scores

print('Funcao rmsle_cv definida (5-fold, embaralhado, seed=42).')

## 4. Modelo 1 — Baseline: Regressão Ridge

Modelo linear com regularização L2. Excelente ponto de partida:  
- Interpreta todas as 251 features de uma vez  
- Lida bem com multicolinearidade (ex: `total_sf` vs `1stFlrSF`)  
- Treinamento instantâneo — serve como régua de comparação

In [ ]:
ridge = Ridge(alpha=10)
scores_ridge = rmsle_cv(ridge, X, y)

scores_str = ', '.join('{:.4f}'.format(s) for s in scores_ridge)
print('Ridge (alpha=10)')
print(f'  RMSLE CV medio:  {scores_ridge.mean():.5f}')
print(f'  Desvio padrao:   +/-{scores_ridge.std():.5f}')
print(f'  Scores por fold: [{scores_str}]')

## 5. Modelo 2 — Random Forest

Ensemble de árvores de decisão treinadas de forma independente (bagging).  
- Captura relações **não-lineares** automaticamente  
- Robusto a outliers e features irrelevantes  
- `max_features='sqrt'` usa ~16 features por split (de 251 disponíveis)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
scores_rf = rmsle_cv(rf, X, y)

scores_str = ', '.join('{:.4f}'.format(s) for s in scores_rf)
print('Random Forest (100 arvores)')
print(f'  RMSLE CV medio:  {scores_rf.mean():.5f}')
print(f'  Desvio padrao:   +/-{scores_rf.std():.5f}')
print(f'  Scores por fold: [{scores_str}]')

## 6. Modelo 3 — Gradient Boosting

Ensemble que constrói árvores **sequencialmente**: cada árvore corrige os erros da anterior.  
- `learning_rate=0.05` + `n_estimators=300`: muitos passos pequenos (mais estável)  
- `subsample=0.8`: amostragem aleatória de 80% dos dados por árvore (evita overfitting)  
- Tende a obter os **melhores resultados em dados tabulares**

> *Pode demorar ~1-2 minutos para o CV.*

In [ ]:
gbm = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=RANDOM_STATE
)
scores_gbm = rmsle_cv(gbm, X, y)

scores_str = ', '.join('{:.4f}'.format(s) for s in scores_gbm)
print('Gradient Boosting (300 estimadores, lr=0.05)')
print(f'  RMSLE CV medio:  {scores_gbm.mean():.5f}')
print(f'  Desvio padrao:   +/-{scores_gbm.std():.5f}')
print(f'  Scores por fold: [{scores_str}]')

## 7. Comparação dos Modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Ridge', 'Random Forest', 'Gradient Boosting'],
    'RMSLE_medio': [scores_ridge.mean(), scores_rf.mean(), scores_gbm.mean()],
    'RMSLE_std':   [scores_ridge.std(),  scores_rf.std(),  scores_gbm.std()],
}).sort_values('RMSLE_medio').reset_index(drop=True)

print(resultados.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
palette = ['#4CAF50', '#2196F3', '#FF9800']
bars = ax.bar(
    resultados['Modelo'], resultados['RMSLE_medio'],
    yerr=resultados['RMSLE_std'], capsize=6,
    color=palette[:len(resultados)], edgecolor='white', alpha=0.85
)
for bar, val in zip(bars, resultados['RMSLE_medio'].values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, val + 0.0005,
        f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax.set_title('Comparacao de Modelos — RMSLE (CV 5-fold)', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSLE (menor e melhor)')
ax.set_ylim(0, resultados['RMSLE_medio'].max() * 1.4)
plt.tight_layout()
plt.show()

## 8. Coeficientes do Modelo — Ridge

O Ridge atribui um peso (coeficiente) a cada feature. Os maiores valores absolutos indicam as features com mais influência na predição do preço.

In [ ]:
# Treinar Ridge com todos os dados para extrair coeficientes
ridge.fit(X, y)

coef = pd.Series(np.abs(ridge.coef_), index=X.columns)
top20 = coef.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Features mais Influentes (Ridge — |coeficiente|)', fontsize=13, fontweight='bold')
ax.set_xlabel('|Coeficiente|')
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(top20.head(10).to_string())

## 9. Salvar o Melhor Modelo

O Ridge treinado em todos os dados é salvo como `modelo_baseline.joblib` na raiz do projeto — é o arquivo que o `pipeline.py` carrega para fazer predições.

In [ ]:
joblib.dump(ridge, '../modelo_baseline.joblib')
print('Modelo salvo em: modelo_baseline.joblib')

# Verificacao rapida nas 5 primeiras amostras
modelo_carregado = joblib.load('../modelo_baseline.joblib')
pred_amostra = np.expm1(modelo_carregado.predict(X[:5]))
real_amostra = np.expm1(y[:5].values)
erros = np.abs(pred_amostra - real_amostra)

print('Verificacao (5 primeiras amostras):')
print(f'  Predito:  {pred_amostra.astype(int)}')
print(f'  Real:     {real_amostra.astype(int)}')
print(f'  Erro abs: {erros.astype(int)}')

## 10. Resumo

| Modelo | RMSLE (CV 5-fold) | Observação |
|--------|-------------------|------------|
| **Ridge** | **0.1105 ± 0.0048** | **Melhor resultado — modelo escolhido** |
| Gradient Boosting | 0.1166 ± 0.0056 | Não-linear, bom resultado |
| Random Forest | 0.1374 ± 0.0063 | Não-linear, robusto |

**Próximos passos:**
1. Ajuste de hiperparâmetros com `GridSearchCV` ou `RandomizedSearchCV`
2. Testar `HistGradientBoostingRegressor` (sklearn) — mais rápido com resultados similares
3. Combinação de modelos (stacking)
4. Executar o `pipeline.py` para gerar predições no `teste.csv`:

```bash
python pipeline.py                         # treina + salva modelo
python pipeline.py --prever data/teste.csv # gera data/predicoes.csv
```